In [1]:
import pandas as pd
from pathlib import Path

In [2]:
#Entrando no caminho dos arquivos
DATA_PATH = Path("../data/raw")

files = list(DATA_PATH.glob("*.csv"))

for file in files:
    print(file.name)

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [3]:
# Carregando os datasets em um dicionário
# CEP é lido como string: como inteiro os zeros à esquerda se perdem (ex.: 01046 -> 1046)
zip_dtypes = {
    "customer_zip_code_prefix": str,
    "seller_zip_code_prefix": str,
    "geolocation_zip_code_prefix": str,
}

datasets = {}

for file in files:
    name = file.stem
    datasets[name] = pd.read_csv(file, dtype=zip_dtypes)

print(f"{len(datasets)} datasets carregados.")

9 datasets carregados.


In [4]:
# Printando a forma de cada dataset
for name, df in datasets.items():
    print(f"{name}: {df.shape}")

olist_customers_dataset: (99441, 5)
olist_geolocation_dataset: (1000163, 5)
olist_orders_dataset: (99441, 8)
olist_order_items_dataset: (112650, 7)
olist_order_payments_dataset: (103886, 5)
olist_order_reviews_dataset: (99224, 7)
olist_products_dataset: (32951, 9)
olist_sellers_dataset: (3095, 4)
product_category_name_translation: (71, 2)


In [5]:
# Carregando dataframes
payments_df = pd.DataFrame(datasets["olist_order_payments_dataset"])
orders_df = pd.DataFrame(datasets["olist_orders_dataset"])
order_items_df = pd.DataFrame(datasets["olist_order_items_dataset"])
payments_df.shape

(103886, 5)

In [6]:
payments_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB


In [7]:
#Avaliando se existe valores nulos
payments_df.isna().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [8]:
#Avaliando se existe dados duplicados
print(f"Linhas totalmente duplicadas: {payments_df.duplicated().sum()}")
print(f"Duplicidade em order_id + payment_sequential: {payments_df.duplicated(subset=['order_id', 'payment_sequential']).sum()}")

Linhas totalmente duplicadas: 0
Duplicidade em order_id + payment_sequential: 0


In [9]:
# Tipos de pagamento (dicionário: credit_card, boleto, voucher, debit_card)
payments_df['payment_type'].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [10]:
payments_df[["payment_installments", "payment_value"]].describe()

,payment_installments,payment_value
count,103886.000000,103886.000000
mean,2.853349,154.100380
std,2.687051,217.494064
min,0.000000,0.000000
25%,1.000000,56.790000
50%,1.000000,100.000000
75%,4.000000,171.837500
max,24.000000,13664.080000


In [11]:
# Frequência de parcelas
payments_df['payment_installments'].value_counts().sort_index()

payment_installments
0         2
1     52546
2     12413
3     10461
4      7098
5      5239
6      3920
7      1626
8      4268
9       644
10     5328
11       23
12      133
13       16
14       15
15       74
16        5
17        8
18       27
20       17
21        3
22        1
23        1
24       18
Name: count, dtype: int64

In [12]:
# Pagamentos com tipo not_defined: a que pedidos pertencem?
not_defined = payments_df[payments_df['payment_type'] == 'not_defined']

print(f"Registros not_defined: {len(not_defined)}")
print(f"Valor pago nesses registros: {not_defined['payment_value'].sum()}")
print(f"Itens em order_items desses pedidos: {order_items_df['order_id'].isin(not_defined['order_id']).sum()}")
orders_df.loc[orders_df['order_id'].isin(not_defined['order_id']), ['order_id', 'order_status']]

Registros not_defined: 3
Valor pago nesses registros: 0.0
Itens em order_items desses pedidos: 0


,order_id,order_status
1130,00b1cb0320190ca0daa2c88b35206009,canceled
39919,4637ca194b6387e2d538dc89b124b0ee,canceled
40235,c8c528189310eaa44a745b8d9d26908b,canceled


In [13]:
# Pagamentos com valor zero, por tipo
payments_df.loc[payments_df['payment_value'] == 0, 'payment_type'].value_counts()

payment_type
voucher        6
not_defined    3
Name: count, dtype: int64

In [14]:
# Pagamentos com 0 parcelas: a que pedidos pertencem?
zero_installments = payments_df[payments_df['payment_installments'] == 0]

print(f"Registros com 0 parcelas: {len(zero_installments)}")
zero_installments.merge(orders_df[['order_id', 'order_status']], on='order_id')

Registros com 0 parcelas: 2


,order_id,payment_sequential,payment_type,payment_installments,payment_value,order_status
0,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69,delivered
1,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94,delivered


In [15]:
# payment_sequential deve ser 1..N dentro de cada pedido
seq = payments_df.groupby('order_id')['payment_sequential'].agg(['count', 'max'])
gaps = seq[seq['count'] != seq['max']]

print(f"Pedidos com buracos na sequência de pagamentos: {len(gaps)}")
print(f"Pedidos com mais de um pagamento: {(seq['count'] > 1).sum()}")

Pedidos com buracos na sequência de pagamentos: 80
Pedidos com mais de um pagamento: 2961


In [16]:
# Integridade com orders
print(f"Pagamentos cujo order_id não existe em orders: {(~payments_df['order_id'].isin(orders_df['order_id'])).sum()}")
print(f"Pedidos sem nenhum pagamento: {(~orders_df['order_id'].isin(payments_df['order_id'])).sum()}")

Pagamentos cujo order_id não existe em orders: 0


Pedidos sem nenhum pagamento: 1


In [17]:
#Fazendo copia do dataframe para não alterar o original
payments_clean = payments_df.copy()

In [18]:
# Parcelas = 0 não existe: o pagamento é à vista, então passa a ser 1
payments_clean.loc[payments_clean['payment_installments'] < 1, 'payment_installments'] = 1

In [19]:
# Verificando quantos valores foram alterados em cada coluna
for col in payments_df.columns:
    print(f"{col}: {(payments_df[col] != payments_clean[col]).sum()} valores alterados")

order_id: 0 valores alterados
payment_sequential: 0 valores alterados
payment_type: 0 valores alterados
payment_installments: 2 valores alterados
payment_value: 0 valores alterados


In [20]:
# Conferindo se o total pago bate com itens + frete (validação cruzada com order_items)
items_total = order_items_df.groupby('order_id')[['price', 'freight_value']].sum().sum(axis=1)
paid_total = payments_clean.groupby('order_id')['payment_value'].sum()

common = items_total.index.intersection(paid_total.index)
diff = (paid_total[common] - items_total[common]).abs()

print(f"Pedidos comparados (com itens e com pagamento): {len(common)}")
print(f"Pedidos com diferença de até R$0,01: {(diff <= 0.01).mean() * 100:.2f}%")
print(f"Total pago: R${paid_total[common].sum():,.2f} | itens + frete: R${items_total[common].sum():,.2f}")

Pedidos comparados (com itens e com pagamento): 98665
Pedidos com diferença de até R$0,01: 99.61%
Total pago: R$15,846,280.17 | itens + frete: R$15,843,409.78


In [21]:
# Validações pós-tratamento
print(f"Linhas: {payments_clean.shape[0]} (original: {payments_df.shape[0]})")
print(f"Valores nulos: {payments_clean.isna().sum().sum()}")
print(f"Parcelas menores que 1: {(payments_clean['payment_installments'] < 1).sum()}")
print(f"Valores de pagamento negativos: {(payments_clean['payment_value'] < 0).sum()}")
print(f"Duplicidade em order_id + payment_sequential: {payments_clean.duplicated(subset=['order_id', 'payment_sequential']).sum()}")
print(f"order_id sem correspondência em orders: {(~payments_clean['order_id'].isin(orders_df['order_id'])).sum()}")

assert payments_clean.shape == payments_df.shape
assert (payments_clean['payment_installments'] >= 1).all()
assert not payments_clean.duplicated(subset=['order_id', 'payment_sequential']).any()

Linhas: 103886 (original: 103886)
Valores nulos: 0
Parcelas menores que 1: 0
Valores de pagamento negativos: 0
Duplicidade em order_id + payment_sequential: 0
order_id sem correspondência em orders: 0


In [22]:
# Salvando dataframe tratado em pasta processed
payments_clean.to_csv('../data/processed/olist_order_payments_dataset_clean.csv', index=False)

## 📌 Conclusão da etapa — Order Payments

### 🔎 O que foi encontrado?

A tabela `order_payments` possui **103.886 registros e 5 colunas**.

Durante o profiling, foram identificados:

- Nenhum valor nulo e nenhuma duplicidade em `order_id + payment_sequential`;
- Mix de pagamentos: `credit_card` (76.795), `boleto` (19.784), `voucher` (5.775), `debit_card` (1.529) e **3 registros `not_defined`**;
- Os 3 `not_defined` pertencem a pedidos **cancelados**, sem itens e com valor 0;
- **9 pagamentos com valor 0** (6 `voucher` e os 3 `not_defined`);
- **2 pagamentos com 0 parcelas**, ambos `credit_card` de pedidos entregues e com valor maior que zero;
- **80 pedidos** com buracos na sequência de `payment_sequential` e **2.961 pedidos** com mais de um pagamento;
- **1 pedido** sem nenhum pagamento.

### 🛠️ O que foi feito?

- `payment_installments = 0` foi alterado para `1` (pagamento à vista), **2 valores alterados**;
- Os registros `not_defined`, de valor zero e os buracos em `payment_sequential` foram **mantidos**, pois não afetam valores de receita e removê-los apagaria o histórico dos pedidos. Para análises de forma de pagamento, é recomendado excluir `not_defined`.

Nenhum registro foi removido.

### ✅ Como foi validado?

- A quantidade de registros permaneceu em **103.886**;
- Sem nulos, sem parcelas menores que 1 e sem valores negativos;
- Todos os `order_id` possuem correspondência em `orders`;
- Validação cruzada com `order_items`: em **98.665 pedidos** com itens e pagamento, **99,61%** têm total pago igual a `price + frete` (diferença de até R$ 0,01). Total pago de R$ 15,85 milhões contra R$ 15,84 milhões em itens + frete.

### 📁 Resultado

O dataset tratado foi salvo em:

`data/processed/olist_order_payments_dataset_clean.csv`

> ⚠️ Como um pedido pode ter vários pagamentos e vários itens, agregar por `order_id` antes de juntar `payments` com `order_items`; caso contrário a receita é duplicada.